# ZIP密码恢复工具 - APK在线构建

本笔记本用于在Google Colab中自动构建Android APK文件。

**使用方法：**
1. 点击上方菜单 `代码执行程序` -> `更改运行时类型` -> 选择 `Python 3`
2. 依次点击每个单元格左侧的 ▶ 按钮运行
3. 最后会自动下载APK文件

⚠️ 构建过程约需30-60分钟，请保持页面打开

## 第一步：安装系统依赖

In [ ]:
!apt-get update -qq
!apt-get install -y -qq build-essential git python3 python3-pip openjdk-17-jdk autoconf libtool pkg-config cmake zip unzip wget ccache
!pip install -q buildozer cython==0.29.33
print('✅ 依赖安装完成')

## 第二步：上传项目文件

运行下方代码后，点击出现的 `选择文件` 按钮，上传以下两个文件：
- `main.py`（Kivy主程序）
- `buildozer.spec`（构建配置）

文件位置：`C:\Users\Administrator\Doubao\chats\zip_cracker\android\`

In [ ]:
import os
os.makedirs('/content/zip_cracker', exist_ok=True)
os.chdir('/content/zip_cracker')

from google.colab import files
uploaded = files.upload()

for fn in uploaded.keys():
    print(f'✅ 已上传: {fn} ({len(uploaded[fn])} bytes)')

print('\n当前目录文件:')
!ls -la /content/zip_cracker/

## 第三步：配置环境变量

In [ ]:
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']
!java -version
print('✅ Java环境配置完成')

## 第四步：开始构建APK

⚠️ 这一步需要30-60分钟，期间会自动下载Android SDK/NDK（约2GB），请耐心等待。

如果中途断开连接，重新运行此单元格即可继续。

In [ ]:
import os
os.chdir('/content/zip_cracker')

# 接受Android SDK许可
!yes | buildozer android sdkmanager --licenses 2>/dev/null || true

# 开始构建
!buildozer android debug 2>&1 | tail -100

## 第五步：检查并下载APK

In [ ]:
import os
import glob
from google.colab import files

os.chdir('/content/zip_cracker')
apk_files = glob.glob('bin/*.apk')

if apk_files:
    for apk in apk_files:
        size_mb = os.path.getsize(apk) / 1024 / 1024
        print(f'✅ APK构建成功: {apk} ({size_mb:.1f} MB)')
        files.download(apk)
else:
    print('❌ 未找到APK文件，请检查上方构建日志')
    print('\nbin目录内容:')
    !ls -la bin/ 2>/dev/null || echo 'bin目录不存在'
    print('\n构建日志最后50行:')
    !tail -50 .buildozer/android/platform/build-*/buildozer.log 2>/dev/null || echo '未找到日志'

## 构建失败排查

如果构建失败，运行下方代码查看详细错误日志：

In [ ]:
# 查看构建日志
!find .buildozer -name '*.log' -exec tail -80 {} \; 2>/dev/null | head -200

# 检查关键依赖
print('\n=== 环境检查 ===')
!python3 --version
!java -version 2>&1
!buildozer --version 2>&1 | head -1